# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


## Hizli Akis

Bu notebook secilen tek adapter icin training, OOD kalibrasyonu, final evaluation ve export akisini sirayla calistirir.

- Genelde sadece `ADAPTER_KEY` degistirin; crop, part, dataset, OOD ve OE yollari bu anahtardan acik sekilde set edilir.
- Runtime dataset beklentisi: `data/prepared_runtime_datasets/<adapter_key>/continual|val|test|ood|oe`.
- `ood` final kanit, `oe` sadece training yardimci negatifleridir; ikisini karistirmayin.
- Push veya publish son adimda bozulursa training bittiyse run klasorunu kurtarmayi deneyin, bastan egitime donmeyin.


In [1]:
from pathlib import Path
import os
import subprocess
import sys
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
NOTEBOOK2_SPARSE_PATHS = (
    'README.md',
    'docs',
    'src',
    'scripts',
    'config',
    'colab_notebooks',
    'requirements.txt',
    'requirements_colab.txt',
    'pyproject.toml',
)

def _build_repo_access_url(repo_url, token):
    token = str(token or '').strip()
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    clone_url = _build_repo_access_url(REPO_URL, os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN'))
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', clone_url, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK2_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


[BOOTSTRAP] GitHub token found.
[BOOTSTRAP] HuggingFace token found.
[BOOTSTRAP] Repo root: /content/bitirmeprojesi
[BOOTSTRAP] Installing requirements_colab.txt...
[BOOTSTRAP] Notebook 2: Continual Adapter Training bootstrap complete.

BOOTSTRAP STATUS
Status: ok
Repo Root: /content/bitirmeprojesi
In Colab: True
GitHub Token: ✓
HuggingFace Token: ✓

[SONRAKI] Parametre ve adapter secimi hucresine gecin; model erisimi ayrica access-check hucresinde dogrulanacak.


In [2]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())


[BOOTSTRAP] run_id=tomato_unspecified_2026-05-07_13-23-38 crop=tomato part=unspecified


In [3]:
# Adapter secimi: genelde sadece bunu degistir.
ADAPTER_KEY = "grape__leaf"  # grape__fruit, grape__leaf, strawberry__fruit, strawberry__leaf, tomato__fruit, tomato__leaf

# Bayesian optimizasyon default olarak acik: onceki run'lardan ayni adapter icin parametre onerisi uygular.
ENABLE_BAYESIAN_OPTIMIZATION = True

# Adapter bazli gorunur defaultlar. ADAPTER_KEY hangi bloğun kullanilacagini secer.
ADAPTER_RECS = {
    "grape__fruit": {"crop": "grape", "part": "fruit", "ood": "data/prepared_runtime_datasets/grape__fruit/ood", "oe": "data/prepared_runtime_datasets/grape__fruit/oe", "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False, "overrides": {"EPOCHS": 32, "BATCH_SIZE": 48, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0}},
    "grape__leaf": {"crop": "grape", "part": "leaf", "ood": "data/prepared_runtime_datasets/grape__leaf/ood", "oe": "data/prepared_runtime_datasets/grape__leaf/oe", "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False, "overrides": {"EPOCHS": 28, "BATCH_SIZE": 64, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.12, "OOD_FACTOR": 3.0}},
    "strawberry__fruit": {"crop": "strawberry", "part": "fruit", "ood": "data/prepared_runtime_datasets/strawberry__fruit/ood", "oe": "data/prepared_runtime_datasets/strawberry__fruit/oe", "oe_enabled": True, "oe_w": 0.10, "allow_under_min": True, "overrides": {"EPOCHS": 36, "BATCH_SIZE": 32, "LEARNING_RATE": 8e-5, "LORA_R": 16, "LORA_ALPHA": 16, "LORA_DROPOUT": 0.18, "OOD_FACTOR": 3.0}},
    "strawberry__leaf": {"crop": "strawberry", "part": "leaf", "ood": "data/prepared_runtime_datasets/strawberry__leaf/ood", "oe": "data/prepared_runtime_datasets/strawberry__leaf/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 22, "BATCH_SIZE": 96, "LEARNING_RATE": 1.5e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0}},
    "tomato__fruit": {"crop": "tomato", "part": "fruit", "ood": "data/prepared_runtime_datasets/tomato__fruit/ood", "oe": "data/prepared_runtime_datasets/tomato__fruit/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 30, "BATCH_SIZE": 64, "LEARNING_RATE": 8e-5, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0}},
    "tomato__leaf": {"crop": "tomato", "part": "leaf", "ood": "data/prepared_runtime_datasets/tomato__leaf/ood", "oe": "data/prepared_runtime_datasets/tomato__leaf/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 20, "BATCH_SIZE": 96, "LEARNING_RATE": 1.2e-4, "LORA_R": 32, "LORA_ALPHA": 32, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0}},
}

# Adapterden bagimsiz gorunur defaultlar.
DEFAULT_RUNTIME_PARAMS = {
    "ENABLE_BAYESIAN_OPTIMIZATION": ENABLE_BAYESIAN_OPTIMIZATION,
    "WEIGHT_DECAY": 0.01,
    "LOSS_NAME": "logitnorm",
    "LOGITNORM_TAU": 1.0,
    "MIXED_PRECISION": "bf16",
    "GRAD_ACCUM_STEPS": 1,
    "MAX_GRAD_NORM": 1.0,
    "LABEL_SMOOTHING": 0.0,
    "SCHEDULER_NAME": "cosine",
    "SCHEDULER_WARMUP_RATIO": 0.1,
    "SCHEDULER_MIN_LR": 1e-6,
    "EARLY_STOPPING_PATIENCE": 6,
    "EARLY_STOPPING_MIN_DELTA": 0.0,
    "DETERMINISTIC": False,
    "SEED": 42,
    "RANDAUGMENT_NUM_OPS": 2,
    "RANDAUGMENT_MAGNITUDE": 7,
    "NUM_WORKERS": 12,
    "PREFETCH": 8,
    "PIN_MEMORY": True,
    "USE_CACHE": True,
    "CACHE_TRAIN_SPLIT": True,
    "VALIDATION_EVERY_N_EPOCHS": 1,
    "CHECKPOINT_EVERY_N_STEPS": 250,
    "CHECKPOINT_ON_EXCEPTION": True,
    "STDOUT_BATCH_INTERVAL": 12,
    "RESUME_MODE": "fresh",
    "AUTO_DISCONNECT_RUNTIME": True,
    "AUTO_DISCONNECT_GRACE_SECONDS": 20,
    "AUTO_PUSH_TO_GITHUB": True,
    "AUTO_PUSH_REMOTE_NAME": "origin",
    "AUTO_PUSH_BRANCH": None,
}

# Sadece bu run icin defaultlari ezmek istersen buraya yaz.
MANUAL_PARAM_OVERRIDES = {}

with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    from scripts.notebook_helpers.cell_script_runner import run_cell_script
    run_cell_script('nb2_cell04_parameter_resolution.py', globals())


[ADAPTER_SELECTED] key=grape__leaf crop=grape part=leaf dataset=grape__leaf ood=data/prepared_runtime_datasets/grape__leaf/ood oe_enabled=True oe=data/prepared_runtime_datasets/grape__leaf/oe run_id=grape_leaf_2026-05-07_13-23-38
[TRAINING_PLAN] epochs=28 batch=64 lr=0.0001 lora_r=24 allow_under_min=False validation_every=1
[SONRAKI] Erisim kontrolu -> dataset validation -> engine init -> training -> calibration -> save/eval hucresini sirayla calistirin.
[BAYES] Toggle acik ama dosya yok: /content/bitirmeprojesi/runs/_index/bayesian_recommendations.json


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=grape epochs=12 bs=96 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=True
[DATASET] runtime_root=data/prepared_runtime_datasets dataset_name=grape__leaf
[OOD] ood_root=data/prepared_runtime_datasets/grape__leaf/ood ask_for_ood_root=False
[OE] oe_root=data/prepared_runtime_datasets/grape__leaf/oe ask_for_oe_root=False enabled=True loss_weight=0.2
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=1 cache_train=True aug=randaugment randaug=2/7 
[OOD] factor=3.0 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [4]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.


In [5]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell06_dataset_validation.py', globals())


[BOOTSTRAP] Fetching selected Notebook 2 dataset paths: ['data/prepared_runtime_datasets/grape__leaf', 'data/ood_dataset/final/grape__leaf_ood_final', 'data/oe_dataset/grape_leaf_oe_unsupported_leaf_candidates', 'data/prepared_runtime_datasets/grape__leaf/ood', 'data/prepared_runtime_datasets/grape__leaf/oe']
[OOD] explicit ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/ood
[OE] explicit oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/oe
[DATASET] runtime root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf classes=7: ['üzüm_antraknoz_yaprak', 'üzüm_kav_esca_yaprak', 'üzüm_külleme_yaprak', 'üzüm_leafroll_virüs_yaprak', 'üzüm_mildiyö_yaprak', 'üzüm_sağlıklı_yaprak', 'üzüm_yelpaze_virüsü_yaprak']
[DATASET][CHECK] scale=small continual=1207 val=324 test=324 ood=263 classes=7
[DATASET][WARN] Manifest rows include 182 eval-quality-risk item(s); prefer cautious regularization.
[PARAMS] Manual overr

In [6]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


[OOD] selected ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/ood
[OE] selected oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf/oe
[CLASSES] ['u_zu_m_antraknoz_yaprak', 'u_zu_m_kav_esca_yaprak', 'u_zu_m_ku_lleme_yaprak', 'u_zu_m_leafroll_viru_s_yaprak', 'u_zu_m_mildiyo_yaprak', 'u_zu_m_sag_lıklı_yaprak', 'u_zu_m_yelpaze_viru_su_yaprak']
[CLASSES] mode=dataset_fallback reason=partial_taxonomy_alignment_fallback matched=0 unmatched=7
[CLASSES] taxonomy-unmatched classes kept: ['u_zu_m_antraknoz_yaprak', 'u_zu_m_kav_esca_yaprak', 'u_zu_m_ku_lleme_yaprak', 'u_zu_m_leafroll_viru_s_yaprak', 'u_zu_m_mildiyo_yaprak', 'u_zu_m_sag_lıklı_yaprak', 'u_zu_m_yelpaze_viru_su_yaprak']
[ENGINE][OOD_CFG] {"ber_enabled": false, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "energy_temperature

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Hazir. trainable_params=13,772,555  classes=7


In [7]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.


In [8]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


[TRAIN] epochs=28 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/18 loss=0.0000 lr=0.000100 speed=255.9s/s elapsed=6s eta=252s
[CKPT] epoch_end epoch=1 step=18
[EPOCH] 1/28: train_loss=1.9899 val_loss=1.9332 val_acc=0.1636 macro_f1=0.0403 * BEST
[TRAIN] epoch=2 batch=12/18 loss=0.0000 lr=0.000099 speed=255.6s/s elapsed=18s eta=287s
[EPOCH] 2/28: train_loss=1.9429 val_loss=1.9394 val_acc=0.0926 macro_f1=0.0550
[TRAIN] epoch=3 batch=12/18 loss=0.0000 lr=0.000098 speed=255.9s/s elapsed=26s eta=245s
[EPOCH] 3/28: train_loss=1.9235 val_loss=1.9721 val_acc=0.1173 macro_f1=0.0796
[TRAIN] epoch=4 batch=12/18 loss=0.0000 lr=0.000096 speed=255.9s/s elapsed=33s eta=221s
[CKPT] epoch_end epoch=4 step=72
[EPOCH] 4/28: train_loss=1.8303 val_loss=1.5488 val_acc=0.4043 macro_f1=0.3585 * BEST
[TRAIN] epoch=5 batch=12/18 loss=0.0000 lr=0.000093 speed=255.9s/s elapsed=44s eta=219s
[CKPT] epoch_end epoch=5 step=90
[EPOCH] 5/28: train_loss=1.6713 val_loss=1.3526 val_acc=0.4630 macro_f1=0.4038 * BES

In [9]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


[OOD] Kalibrasyon tamamlandi. classes=7 version=1


In [10]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


Cell 8: adapter save started
Kaydedilen adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/fusion.pth
Telemetry adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/telemetry_runtime/telemetry/grape_leaf_2026-05-07_13-23-38/artifacts/adapter_export/grape/leaf/continual_sd_lora_adapter
Cell 8: adapter save completed


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell12_final_evaluation.py', globals())


[DOGRULAMA (referans)] ornek=324 sinif=7 accuracy=0.8272 ood_auroc=0.8031 sure_ds_f1=0.3253 conformal_cov=0.9537
[TEST (ayrilmis)] ornek=324 sinif=7 accuracy=0.8735 ood_auroc=0.8018 sure_ds_f1=0.3275 conformal_cov=0.9660
[OOD] kanit=real_ood_split durum=failed gecti=False
[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.
